# PARP-1 Inhibitor Classifier

Predicts whether a compound inhibits human PARP-1, trained on measured IC50, Ki and Kd values
from BindingDB and ChEMBL.

This notebook trains the model from scratch, evaluates it two different ways, saves it, and
runs predictions on new compounds. Run the cells in order.

**What you need:** `cleaned_dataset.csv`, which is the curated training set of 5,587 compounds.

## How the model works, in short

Each molecule is cleaned the same way every time, which means salts stripped, charges
neutralised and tautomers put into a single canonical form. That last step matters more than it
sounds: the same molecule can be drawn two ways, and a fingerprint treats those as unrelated
structures. Fixing it was the single biggest improvement we found.

The molecule is then turned into a 2048-bit Morgan fingerprint, which is a list of yes or no
answers about which small fragments are present. An Extra Trees ensemble of 500 decision trees
learns which fragment patterns go with activity.

Active means an IC50 below 1 micromolar. Inactive means 10 micromolar or above. Compounds
measured between those two values are excluded, which follows the rule used by the PARP1pred
benchmark study.


## 1. Setup


In [ ]:
!pip -q install rdkit

import json, os, warnings
from collections import defaultdict

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, balanced_accuracy_score,
                             confusion_matrix, matthews_corrcoef, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedShuffleSplit
from scipy import stats

warnings.filterwarnings('ignore')
RDLogger.DisableLog('rdApp.*')
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
print('ready')


## 2. Load the dataset

Upload `cleaned_dataset.csv` when prompted, or mount Drive and point `DATA` at it.


In [ ]:
DATA = 'cleaned_dataset.csv'

if not os.path.exists(DATA):
    try:
        from google.colab import files
        print('Upload cleaned_dataset.csv')
        up = files.upload()
        DATA = list(up.keys())[0]
    except ImportError:
        raise SystemExit('Put cleaned_dataset.csv next to this notebook.')

raw = pd.read_csv(DATA)
print(f'rows: {len(raw)}')
print(raw.activity.value_counts().to_string())
raw.head(3)


## 3. Prepare the data

We drop the excluded middle band, since the model only learns from compounds that are clearly
active or clearly inactive. Then every molecule becomes a fingerprint.


In [ ]:
d = raw[raw.label != -1].reset_index(drop=True)
d['mol'] = [Chem.MolFromSmiles(s) for s in d.smiles]
d = d[d.mol.notna()].reset_index(drop=True)

y = d.label.values
scaffolds = d.scaffold.fillna('').values

morgan3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
morgan2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)   # for similarity
X = np.stack([morgan3.GetFingerprintAsNumPy(m) for m in d.mol]).astype(np.float32)

print(f'compounds: {len(d)}   active: {(y == 1).sum()}   inactive: {(y == 0).sum()}')
print(f'fingerprint matrix: {X.shape}')
print(f'distinct scaffolds: {len(set(scaffolds))}')


## 4. Two ideas that matter

### Scaffold splitting

A random split puts close analogues of a test compound into the training set, so the model can
score well just by recognising near copies. Since the point is to predict compounds nobody has
made yet, that flatters the model. Scaffold splitting groups compounds by their core framework
and sends whole groups to one side of the split, so the test set always asks about chemistry
the model has never seen.

### Scaffold-inverse weighting

The training data is dominated by a few large chemical families, especially the olaparib-like
series. Left alone, the model mostly learns whichever fragments are common in those families.
Weighting each compound by 1 over the square root of its scaffold group size stops any single
series from driving the result. We combine that with class balancing, because the data is about
88 percent active.


In [ ]:
def sample_weights(idx):
    """Balance the classes, then damp down over-represented scaffolds."""
    group_size = defaultdict(int)
    for i in idx:
        group_size[scaffolds[i]] += 1
    w = np.ones(len(idx), dtype=float)
    for cls in (0, 1):
        mask = y[idx] == cls
        if mask.sum():
            w[mask] = len(idx) / (2.0 * mask.sum())
    return w * np.array([1.0 / np.sqrt(group_size[scaffolds[i]]) for i in idx])


def scaffold_split(seed, test_fraction=0.2):
    """Whole scaffold groups go to one side or the other, never split across both."""
    groups = defaultdict(list)
    for i, s in enumerate(scaffolds):
        groups[s].append(i)
    groups = list(groups.values())
    n = len(d)
    rng = np.random.RandomState(seed)
    large = [g for g in groups if len(g) > 0.1 * n]
    small = [g for g in groups if len(g) <= 0.1 * n]
    rng.shuffle(large); rng.shuffle(small)
    train, test = [], []
    for g in large + small:
        (train if len(train) + len(g) <= (1 - test_fraction) * n else test).extend(g)
    return np.array(train), np.array(test)

tr, te = scaffold_split(0)
print(f'example split: {len(tr)} train, {len(te)} test')
print(f'scaffolds shared between the two halves: '
      f'{len(set(scaffolds[tr]) & set(scaffolds[te]))}   (should be 0)')


## 5. Cross validation

Ten scaffold splits. For each one we train on 80 percent, predict the rest, and pick the
decision threshold that balances performance on actives and inactives.

This takes a couple of minutes.


In [ ]:
N_SPLITS, N_TREES = 10, 500
THRESHOLDS = np.arange(0.05, 0.96, 0.005)

oof = np.full(len(d), np.nan)      # out-of-fold predictions
per_split = []

for seed in range(N_SPLITS):
    train, test = scaffold_split(seed)
    clf = ExtraTreesClassifier(n_estimators=N_TREES, n_jobs=-1, random_state=seed)
    clf.fit(X[train], y[train], sample_weight=sample_weights(train))
    p = clf.predict_proba(X[test])[:, 1]
    oof[test] = p
    cut = float(max(THRESHOLDS,
                    key=lambda t: balanced_accuracy_score(y[test], (p >= t).astype(int))))
    pred = (p >= cut).astype(int)
    per_split.append(dict(seed=seed, cutoff=cut,
                          MCC=matthews_corrcoef(y[test], pred),
                          BA=balanced_accuracy_score(y[test], pred),
                          AUC=roc_auc_score(y[test], p)))
    print(f'  split {seed}: MCC {per_split[-1]["MCC"]:.3f}   cutoff {cut:.3f}')

splits = pd.DataFrame(per_split)
print(f'\nMCC {splits.MCC.mean():.3f} +/- {splits.MCC.std():.3f}')
print(f'balanced accuracy {splits.BA.mean():.3f}')
print(f'ROC-AUC {splits.AUC.mean():.3f}')


## 6. Results

The threshold is chosen on the pooled out-of-fold predictions, which means it is never fitted
on data the final model will also be trained on.


In [ ]:
seen = ~np.isnan(oof)
cutoff = float(max(THRESHOLDS,
                   key=lambda t: balanced_accuracy_score(y[seen], (oof[seen] >= t).astype(int))))
pred = (oof[seen] >= cutoff).astype(int)
cm = confusion_matrix(y[seen], pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print(f'decision threshold: {cutoff:.3f}')
print(f'MCC               {matthews_corrcoef(y[seen], pred):.3f}')
print(f'balanced accuracy {balanced_accuracy_score(y[seen], pred):.3f}')
print(f'specificity       {tn / (tn + fp):.3f}   (inactives correctly rejected)')
print(f'recall            {tp / (tp + fn):.3f}   (actives correctly found)')
print(f'ROC-AUC           {roc_auc_score(y[seen], oof[seen]):.3f}')

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay(cm, display_labels=['Inactive', 'Active']).plot(
    ax=ax[0], cmap='Blues', colorbar=False, values_format='d')
ax[0].set_title('Out-of-fold confusion matrix'); ax[0].grid(False)
fpr, tpr, _ = roc_curve(y[seen], oof[seen])
ax[1].plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc_score(y[seen], oof[seen]):.3f}')
ax[1].plot([0, 1], [0, 1], '--', color='grey', lw=1)
ax[1].set_xlabel('False positive rate'); ax[1].set_ylabel('True positive rate')
ax[1].set_title('ROC'); ax[1].legend(loc='lower right')
plt.tight_layout(); plt.show()


## 7. Comparison with the published benchmark

PARP1pred reports an MCC of 0.74 on its external test set, and that test set was a random
split. To compare like for like we have to use the same protocol. This cell runs random
splitting so the two numbers mean the same thing.

Expect the random split to score higher than the scaffold split. That gap is not a mistake, it
is the whole point: random splitting is an easier test.


In [ ]:
rows = []
for k, (train, test) in enumerate(StratifiedShuffleSplit(
        n_splits=10, test_size=0.2, random_state=0).split(X, y)):
    w = np.ones(len(train))
    for cls in (0, 1):
        m = y[train] == cls
        w[m] = len(train) / (2.0 * m.sum())
    clf = ExtraTreesClassifier(n_estimators=N_TREES, n_jobs=-1, random_state=k)
    clf.fit(X[train], y[train], sample_weight=w)
    p = clf.predict_proba(X[test])[:, 1]
    cut = float(max(THRESHOLDS,
                    key=lambda t: balanced_accuracy_score(y[test], (p >= t).astype(int))))
    rows.append(matthews_corrcoef(y[test], (p >= cut).astype(int)))

rnd = np.array(rows)
lo, hi = stats.t.interval(0.95, len(rnd) - 1, loc=rnd.mean(), scale=rnd.std(ddof=1) / np.sqrt(len(rnd)))
print(f'random split   MCC {rnd.mean():.3f} +/- {rnd.std(ddof=1):.3f}   95% CI [{lo:.3f}, {hi:.3f}]')
print(f'scaffold split MCC {splits.MCC.mean():.3f} +/- {splits.MCC.std():.3f}')
print(f'\nPARP1pred benchmark: 0.74 (random split, external test set)')
print('Compare our random split number against it. The scaffold number is the honest figure')
print('for how the model behaves on chemistry it has never seen.')


## 8. Train the final model and save it

Now we train on everything, and save the files needed to make predictions later.


In [ ]:
final = ExtraTreesClassifier(n_estimators=N_TREES, n_jobs=-1, random_state=0)
final.fit(X, y, sample_weight=sample_weights(np.arange(len(d))))

joblib.dump(final, 'parp1_extratrees.joblib', compress=3)
pd.DataFrame({'smiles': d.smiles, 'label': y, 'scaffold': scaffolds}).to_csv(
    'training_compounds.csv', index=False)
pd.DataFrame({'score': oof[seen], 'label': y[seen]}).to_csv('oof_scores.csv', index=False)

meta = dict(
    algorithm=f'ExtraTreesClassifier, {N_TREES} trees, scaffold-inverse sample weighting',
    features='Morgan fingerprint, radius 3, 2048 bits',
    training_compounds=int(len(d)), active=int((y == 1).sum()), inactive=int((y == 0).sum()),
    decision_threshold=round(cutoff, 3), conformal_significance=0.20,
    scaffold_MCC=round(float(splits.MCC.mean()), 3),
    random_MCC=round(float(rnd.mean()), 3))
json.dump(meta, open('model_metadata.json', 'w'), indent=2)
print(json.dumps(meta, indent=2))


## 9. Teaching the model to say "I don't know"

A plain classifier is forced to answer every question, even about a compound unlike anything it
has seen. Conformal prediction fixes that. It compares a new score against how known actives
and known inactives scored on held-out data, and only commits when one of the two labels is
statistically incompatible with the score. When both are still possible, it answers Uncertain.

Among the compounds it does decide, it is designed to be wrong at most 20 percent of the time.


In [ ]:
EPS = 0.20                         # allowed error rate on the compounds it decides
CAL_ACTIVE = np.sort(oof[seen][y[seen] == 1])
CAL_INACTIVE = np.sort(oof[seen][y[seen] == 0])


def conformal(score):
    """How plausible is each label for this score?"""
    p_act = (np.searchsorted(CAL_ACTIVE, score, side='right') + 1) / (len(CAL_ACTIVE) + 1)
    p_ina = (len(CAL_INACTIVE) - np.searchsorted(CAL_INACTIVE, score, side='left') + 1) / (len(CAL_INACTIVE) + 1)
    plausible = [lab for lab, pv in (('Active', p_act), ('Inactive', p_ina)) if pv > EPS]
    return p_act, p_ina, (plausible[0] if len(plausible) == 1 else 'Uncertain')


for s in (0.05, 0.4, 0.7, 0.9, 0.99):
    pa, pi, v = conformal(s)
    print(f'score {s:.2f}  ->  {v:<10s} (plausibility: active {pa:.2f}, inactive {pi:.2f})')


## 10. Predict new compounds

Put your SMILES in the list below. Every prediction comes back with four things:

- **score**, the model's actual output, a probability between 0 and 1
- **prediction**, that score compared against the fixed threshold
- **verdict**, Active, Inactive, or Uncertain
- **reliability**, based on how similar the compound is to the training set

Read the reliability column first. Below 0.5 similarity the model's balanced accuracy falls
from about 0.93 to about 0.64, so those predictions are a ranking, not an answer.


In [ ]:
SMILES = [
    'NC(=O)c1cccc2[nH]c(C3(C)CCCN3)nc12',              # veliparib, a known inhibitor
    'O=C(N1CCOCC1)C2=CC(OCCCC)=NC3=CC=CC=C32',          # 8F
    'O=C(N1CCCCC1)C(C2=CC=CC=C2N3CCCC)=CC3=O',          # 16F
    'CCO',                                              # ethanol, a sanity check
]

chooser = rdMolStandardize.LargestFragmentChooser()
uncharger = rdMolStandardize.Uncharger()
normalizer = rdMolStandardize.Normalizer()
taut = rdMolStandardize.TautomerEnumerator(); taut.SetMaxTautomers(200)
TRAIN_FP = [morgan2.GetFingerprint(m) for m in d.mol]


def prepare(smiles):
    """Exactly the cleaning the training data went through."""
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        return None
    try:
        m = taut.Canonicalize(uncharger.uncharge(normalizer.normalize(chooser.choose(m))))
    except Exception:
        pass
    return m


def predict(smiles_list):
    rows = []
    for s in smiles_list:
        m = prepare(s)
        if m is None:
            rows.append(dict(smiles=s, score=np.nan, prediction='could not parse'))
            continue
        x = morgan3.GetFingerprintAsNumPy(m).astype(np.float32).reshape(1, -1)
        score = float(final.predict_proba(x)[0, 1])
        sim = float(max(DataStructs.BulkTanimotoSimilarity(morgan2.GetFingerprint(m), TRAIN_FP)))
        _, _, verdict = conformal(score)
        rows.append(dict(smiles=Chem.MolToSmiles(m), score=round(score, 3),
                         prediction='Active' if score >= cutoff else 'Inactive',
                         verdict=verdict, similarity=round(sim, 3),
                         reliability='High' if sim >= 0.7 else 'Medium' if sim >= 0.5 else 'Low'))
    return pd.DataFrame(rows)


results = predict(SMILES)
print(results.to_string(index=False))


### Predict from a CSV file

If you have a file with a `smiles` column, use this instead.


In [ ]:
# from google.colab import files
# up = files.upload()
# src = pd.read_csv(list(up.keys())[0])
# out = predict(src.smiles.tolist())
# out.to_csv('predictions.csv', index=False)
# files.download('predictions.csv')
# out


## 11. Things to be aware of

**The score is not a probability of being right.** It is the fraction of trees voting active.
A score of 0.7 does not mean a 70 percent chance the compound works.

**Scaffold and random splits measure different things.** The random number is the one to
compare against published benchmarks, because that is how they were measured. The scaffold
number is the one that tells you how the model will behave on a new series.

**The training data is about 88 percent active.** Inactive compounds are the scarce class, so
the model has seen far fewer examples of what inactivity looks like.

**There is a hole in the middle of the data.** Compounds measured between 1 and 10 micromolar
were excluded, so the model has never seen a moderately active compound. Asked about one, it
has no category for it.

**The model matches fragments, it does not understand binding.** On an unfamiliar scaffold it
falls back on whatever fragment it does recognise, and that fragment may not be the part of the
molecule that actually matters. This is why the similarity column deserves more attention than
the score.


## 12. Export the model for the InhibitAI app

The app's server loads one file, `parp1_bundle.joblib`, holding everything section 10 uses: the
trained model, the decision threshold, the out-of-fold scores the conformal step compares against,
and the training compounds for the similarity check.

It also stores a few SMILES with the fingerprints `prepare()` gave them here. When the server
starts, it prepares those SMILES its own way and refuses to run if any fingerprint differs, so the
app can never quietly clean molecules differently from this notebook.

You need `export_model_bundle.py` from `backend/scripts/` in the app repository. Upload it when
prompted. Run sections 1 to 10 first.

**Change `MODEL_VERSION` whenever you retrain on different data.** The server caches results per
model version, so an unchanged version would keep serving the old model's answers.

In [ ]:
MODEL_VERSION = 'ET v1.0'

try:
    from export_model_bundle import export_bundle
except ImportError:
    from google.colab import files
    print('Upload export_model_bundle.py (backend/scripts/ in the app repository)')
    files.upload()
    from export_model_bundle import export_bundle

# Reference compounds for the server's consistency check: the examples from section 10, a few
# variants that make prepare() do real work, and 300 training compounds.
variants = [
    'OC1=NN=C(Cc2ccc(F)c(c2)C(=O)N2CCN(CC2)C(=O)C2CC2)c2ccccc12.Cl',   # olaparib, HCl salt, other tautomer
    'CC(=O)Oc1ccccc1C(=O)[O-].[Na+]',                                   # a carboxylate salt
    'C[C@@]1(CCCN1)c1nc2c(cccc2[nH]1)C(N)=O',                           # a stereocentre
]
ref_smiles = list(SMILES) + variants + d.smiles.sample(300, random_state=0).tolist()
ref_mols = [prepare(s) for s in ref_smiles]
ref_smiles = [s for s, m in zip(ref_smiles, ref_mols) if m is not None]
ref_X = np.stack([morgan3.GetFingerprintAsNumPy(m) for m in ref_mols if m is not None]).astype(np.float32)

export_bundle(
    'parp1_bundle.joblib',
    model_version=MODEL_VERSION,
    classifiers=final,
    threshold=round(cutoff, 3),               # the same value model_metadata.json records
    featurizer={'kind': 'morgan', 'radius': 3, 'n_bits': 2048},
    cal_scores=oof[seen],                     # what conformal() compares against
    cal_labels=y[seen],
    significance=EPS,
    train_smiles=d.smiles.tolist(),           # TRAIN_FP in section 10 (Morgan radius 2)
    applicability_min_similarity=0.5,         # reliability: Low below 0.5
    applicability_high_similarity=0.7,        # High from 0.7
    reference_smiles=ref_smiles,
    reference_features=ref_X,
    metrics=dict(scaffold_MCC=round(float(splits.MCC.mean()), 3), random_MCC=round(float(rnd.mean()), 3),
                 decision_threshold=round(cutoff, 3), training_compounds=int(len(d))),
)

import sklearn, rdkit
print(f'\nThe server needs the same versions:  scikit-learn=={sklearn.__version__}  rdkit=={rdkit.__version__}')

try:
    from google.colab import files
    files.download('parp1_bundle.joblib')
except ImportError:
    pass   # running locally: the file is next to the notebook